# Gen Z Slang Steering — v3 (working)

See the companion theory doc for a full explanation of what went wrong in v1/v2 and why this works.

In [1]:
%%capture
!pip install transformer_lens einops jaxtyping colorama

In [2]:
import torch, einops, functools, gc
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
from tqdm import tqdm
from torch import Tensor
from typing import List
from sklearn.decomposition import PCA
from jaxtyping import Float, Int
from transformer_lens import HookedTransformer, utils as tl_utils
from transformer_lens.hook_points import HookPoint
from colorama import Fore

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

2026-04-16 19:13:59.805665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776366839.832625     824 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776366839.841264     824 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776366839.863547     824 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776366839.863569     824 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776366839.863572     824 computation_placer.cc:177] computation placer alr

Device: cuda


## 1. Load Model

In [ ]:
from huggingface_hub import login
import os

# Set HF_TOKEN env var before running, or paste your token here temporarily
from huggingface_hub import login
import os

login(token=os.getenv("HF_TOKEN"))

# CRITICAL: use from_pretrained_no_processing (matches refusal demo) and set left padding
model = HookedTransformer.from_pretrained_no_processing(
    'google/gemma-2-2b-it',
    device=DEVICE,
    dtype=torch.float16,
    default_padding_side='left',
)
model.tokenizer.padding_side = 'left'
if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token

N_LAYERS = model.cfg.n_layers   # 26
D_MODEL  = model.cfg.d_model    # 2304
print(f'Layers={N_LAYERS}, d_model={D_MODEL}')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
Layers=26, d_model=2304


## 2. Tokenisation helpers

**LEFT padding** is essential. It ensures that `pos=-1` (the last token) refers to the same *semantic* token — the final `\n` after `<start_of_turn>model` — across all sequences regardless of their length. With right-padding, `pos=-1` would be a PAD token for short sequences.

In [4]:
TEMPLATE = '<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n'

def tokenize(instructions: List[str]) -> Int[Tensor, 'batch seq']:
    """Tokenise a list of raw instructions into the Gemma chat format, left-padded."""
    prompts = [TEMPLATE.format(instruction=inst) for inst in instructions]
    return model.tokenizer(
        prompts,
        padding=True,
        truncation=False,
        return_tensors='pt'
    ).input_ids.to(DEVICE)

GENZ_SLANG = [
    'no cap', 'fr fr', 'slay', 'slaps', 'bussin', 'lowkey', 'highkey',
    'understood the assignment', 'it hits different', 'main character',
    'rent free', 'that era', 'understood', 'periodt', 'based', 'mid',
    'rizz', 'sus', 'bet', 'vibe', 'hits different', 'giving', 'ate',
    'left no crumbs', 'the way', 'not me', "i'm dead", 'iconic',
    'living for this', 'obsessed', 'unalived', 'delulu', 'snatched',
    'no thoughts', "it's giving", 'core', 'era', 'ate that',
]

def count_slang(text: str) -> int:
    t = text.lower()
    return sum(t.count(s) for s in GENZ_SLANG)

def strip_prompt(full_output: str) -> str:
    """Strip the user/model turn markers; return only what the model generated."""
    if '<start_of_turn>model' in full_output:
        return full_output.split('<start_of_turn>model')[-1].strip()
    return full_output.strip()

## 3. Generation with hooks

**Critical design choice:** we implement generation **manually** (token-by-token), applying hooks at *every forward pass*. This mirrors the refusal demo exactly and avoids relying on `model.generate()` preserving hook state across steps.

In [5]:
EOS_TOKEN_ID = model.tokenizer.eos_token_id

def generate_with_hooks(
    model: HookedTransformer,
    toks: Int[Tensor, 'batch seq'],
    max_new_tokens: int = 120,
    fwd_hooks: list = [],
) -> List[str]:
    """
    Manual autoregressive generation with hooks active on every forward pass.
    Pre-allocates a fixed GPU buffer — no growing tensors, no logit accumulation.
    Returns decoded strings (only newly generated tokens).
    """
    batch, prompt_len = toks.shape
    # Pre-allocate full buffer on GPU once — avoids torch.cat allocations each step
    buf = torch.zeros((batch, prompt_len + max_new_tokens), dtype=torch.long, device=DEVICE)
    buf[:, :prompt_len] = toks
    finished = torch.zeros(batch, dtype=torch.bool)

    for i in range(max_new_tokens):
        with model.hooks(fwd_hooks=fwd_hooks):
            logits = model(buf[:, :prompt_len + i])
        next_tok = logits[:, -1, :].argmax(dim=-1)   # stays on GPU for buf write
        del logits                                     # free immediately — vocab is 256k!
        buf[:, prompt_len + i] = next_tok
        finished |= next_tok.cpu() == EOS_TOKEN_ID
        if finished.all():
            break

    return model.tokenizer.batch_decode(
        buf[:, prompt_len:].cpu(), skip_special_tokens=True
    )


def get_completions(
    model: HookedTransformer,
    instructions: List[str],
    fwd_hooks: list = [],
    max_new_tokens: int = 120,
    batch_size: int = 1,   # 🔥 keep at 1; 2B model + long prompts fills VRAM fast
) -> List[str]:
    out = []
    for i in tqdm(range(0, len(instructions), batch_size)):
        toks = tokenize(instructions[i:i+batch_size])
        out.extend(generate_with_hooks(model, toks, max_new_tokens, fwd_hooks))
        del toks
        torch.cuda.empty_cache()
    return out

## 4. Datasets

**Key principle:** the *instructions* themselves are **completely neutral** — no mention of slangs, style, or tone in either set. What differs is a **system-level framing** injected via the instruction text itself.

We use two strategies:
- `slang_instructions`: the same questions but with a brief style primer in the instruction ("using slangs in your answer")
- `plain_instructions`: the same questions with a plain style primer ("in plain text")

The critical difference from v1: the question text is **identical** across both sets. Only the style qualifier changes, and it is kept very short so the direction is dominated by output-style activations, not question vocabulary.

In [6]:
TOPICS = [
    'the water cycle', 'how plants grow', 'the solar system', 'machine learning',
    'cooking pasta', 'exercise benefits', 'climate change', 'the internet',
    'photosynthesis', 'black holes', 'the French Revolution', 'renewable energy',
    'DNA and genetics', 'the stock market', 'ocean currents', 'ancient Egypt',
    'artificial intelligence', 'volcanoes', 'the human brain', 'cryptocurrency',
    'space exploration', 'vaccines', 'music theory', 'quantum computing',
    'biodiversity', "Newton's laws", 'plate tectonics', 'the Renaissance',
    'evolution', 'cloud computing', 'the digestive system', 'democracy',
]

# Identical question structure — style qualifier is the ONLY difference
slang_instructions = [f"Explain {t}. Respond using Gen Z slang and internet speak." for t in TOPICS]
plain_instructions  = [f'Explain {t}. Use formal, plain English with no slang.' for t in TOPICS]

print(f'{len(TOPICS)} instruction pairs')
print('Slang:', slang_instructions[0])
print('Plain:', plain_instructions[0])

32 instruction pairs
Emoji: Explain the water cycle. Use emojis in your response.
Plain: Explain the water cycle. Use plain text only, no emojis.


### Sanity check — confirm the model actually uses slangs when asked

In [7]:
check_s = get_completions(model, slang_instructions[:4], max_new_tokens=80)
check_p = get_completions(model, plain_instructions[:4], max_new_tokens=80)

for i in range(2):
    print(f'SLANG [{count_slang(check_s[i])}]: {check_s[i][:200]}')
    print(f'PLAIN [{count_slang(check_p[i])}]: {check_p[i][:200]}')
    print()

100%|██████████| 4/4 [00:34<00:00,  8.58s/it]

EMOJI [19]: 💧🌎☀️🌊💧🌎☀️🌊💧🌎☀️🌊💧🌎☀️🌊

The water cycle is a continuous journey of water on, above, and below the surface of the Earth! 🌍

1. **Evaporation:** ☀️  The sun's heat turns water from lakes, rivers, and ocea
PLAIN [0]: The water cycle is the continuous movement of water on, above, and below the surface of the Earth. It's a natural process that involves several key stages:

1. **Evaporation:**  The sun's heat turns l

EMOJI [10]: 🌱  Plants start their lives as tiny seeds 🫘, which contain all the instructions they need to grow! 🧬

☀️  They need sunlight 🌞 to make their own food through a process called photosynthesis. 🌿  This p
PLAIN [0]: Plants grow through a process called photosynthesis. They take in sunlight, water, and carbon dioxide from the air. Inside their leaves, they use chlorophyll to convert sunlight into energy. This ener



## 5. Extract activations and compute the slang direction

**What the refusal demo does that we now replicate:**
- `run_with_cache` on the full token batch
- Extract `resid_pre` (not `resid_post`) at `pos=-1` — left-padding ensures this is always the last real token
- Pick a **single layer** for the direction (the one with highest separation)

Why `resid_pre`? It captures the residual stream *before* that layer's attention and MLP have modified it, making the signal cleaner and more stable.

In [8]:
def get_mean_activations(
    model: HookedTransformer,
    instructions: List[str],
    pos: int = -1,
    batch_size: int = 2,   # 🔥 low: each batch caches 26 layers × seq_len × 2304 on GPU
) -> dict:   # layer -> Tensor(n, d_model)
    """
    Run model with cache, extract resid_pre at `pos` for all layers.
    Only stores the single position slice — never holds full (batch, seq, d_model) tensors.
    Returns CPU float32 tensors.
    """
    all_acts = {l: [] for l in range(model.cfg.n_layers)}

    for i in tqdm(range(0, len(instructions), batch_size), desc='Caching'):
        toks = tokenize(instructions[i:i+batch_size])
        with torch.no_grad():
            _, cache = model.run_with_cache(
                toks,
                names_filter=lambda n: 'resid_pre' in n,
                return_type=None,   # don't compute/return logits — saves a full (batch, seq, vocab) tensor
            )
        for l in range(model.cfg.n_layers):
            # Slice pos immediately so we never hold (batch, seq_len, d_model) in Python
            all_acts[l].append(cache['resid_pre', l][:, pos, :].detach().cpu().float())
        del cache, toks
        gc.collect()
        torch.cuda.empty_cache()

    return {l: torch.cat(v, 0) for l, v in all_acts.items()}


In [9]:
print('Caching slang activations...')
slang_acts = get_mean_activations(model, slang_instructions)
print('Caching plain activations...')
plain_acts = get_mean_activations(model, plain_instructions)

# Force full GPU flush before any generation
gc.collect()
torch.cuda.empty_cache()
print('Done. Shape:', slang_acts[0].shape)
print(f'GPU free after caching: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


Caching emoji activations...


Caching: 100%|██████████| 16/16 [00:07<00:00,  2.21it/s]


Caching plain activations...


Caching: 100%|██████████| 16/16 [00:07<00:00,  2.20it/s]


Done. Shape: torch.Size([32, 2304])
GPU free after caching: 6.84 GB


In [10]:
# Per-layer L2 separation — tells us where the emoji signal lives
sep = {l: (slang_acts[l].mean(0) - plain_acts[l].mean(0)).norm().item() for l in range(N_LAYERS)}

fig = go.Figure(go.Scatter(x=list(sep.keys()), y=list(sep.values()), mode='lines+markers',
                           marker=dict(color='#f59e0b', size=8)))
fig.update_layout(title='L2 separation per layer (slang vs plain, resid_pre)',
                  xaxis_title='Layer', yaxis_title='||μ_emoji - μ_plain||',
                  template='plotly_white')
fig.show()

BEST_LAYER = max(sep, key=sep.get)
print(f'Best layer: {BEST_LAYER}  (sep={sep[BEST_LAYER]:.2f})')

Best layer: 25  (sep=487.23)


In [11]:
# Compute the slang direction AT THE SINGLE BEST LAYER (not averaged across all layers)
diff = slang_acts[BEST_LAYER].mean(0) - plain_acts[BEST_LAYER].mean(0)
SLANG_DIR = (diff / diff.norm()).to(DEVICE)  # float32, on DEVICE

print(f'Slang direction computed at layer {BEST_LAYER}')
print(f'Shape: {SLANG_DIR.shape}, norm: {SLANG_DIR.norm():.4f}')

Emoji direction computed at layer 25
Shape: torch.Size([2304]), norm: 1.0000


## 6. PCA — confirm the direction separates the classes

In [12]:
top_layers = sorted(sep, key=sep.get, reverse=True)[:10]
n_cols = 5
n_rows = 2
fig = sp.make_subplots(rows=n_rows, cols=n_cols,
                       subplot_titles=[f'L{l} sep={sep[l]:.1f}' for l in sorted(top_layers)],
                       horizontal_spacing=0.05, vertical_spacing=0.15)
for idx, l in enumerate(sorted(top_layers)):
    r, c = idx // n_cols + 1, idx % n_cols + 1
    Xa = slang_acts[l].numpy(); Xb = plain_acts[l].numpy()
    Xp = PCA(2).fit_transform(np.concatenate([Xa, Xb]))
    na = Xa.shape[0]; show = (idx == 0)
    fig.add_trace(go.Scatter(x=Xp[:na,0], y=Xp[:na,1], mode='markers',
        marker=dict(color='#f59e0b', size=7), name='Slang',
        showlegend=show, legendgroup='S'), row=r, col=c)
    fig.add_trace(go.Scatter(x=Xp[na:,0], y=Xp[na:,1], mode='markers',
        marker=dict(color='#3b82f6', size=7), name='Plain',
        showlegend=show, legendgroup='P'), row=r, col=c)
fig.update_layout(height=400, title='PCA (top-10 most separable layers, slang vs plain)',
                  template='plotly_white')
fig.show()

## 7. Hook function

The refusal demo hooks **three** points per layer: `resid_pre`, `resid_mid`, `resid_post`. This ensures the direction is ablated from the stream *entering* each attention block, *between* attention and MLP, and *exiting* the MLP — a complete surgical removal.

In [13]:
def make_steer_hook(direction, coeff, mode='add'):
    d = direction / direction.norm()

    def hook_fn(activation, hook):
        d_local = d.to(device=activation.device, dtype=activation.dtype)

        if mode == 'add':
            activation[:, -1, :] += coeff * d_local
        elif mode == 'subtract':
            activation[:, -1, :] -= coeff * d_local
        elif mode == 'ablate':
            # Remove the component of the activation that lies along the direction
            proj = (activation[:, -1, :] @ d_local)[:, None] * d_local
            activation[:, -1, :] -= proj
        else:
            raise ValueError(f"Unknown mode: {mode!r}")

        return activation

    return hook_fn

def make_hooks(direction, coeff, mode, layers=None):
    if layers is None:
        # 🔥 FIX: apply steering to ALL layers, not just the best one.
        # The direction was computed from layer 25, but steering only works
        # when applied across all layers (see section 11).
        layers = list(range(N_LAYERS))
        
    fn = make_steer_hook(direction, coeff, mode)
    
    # Use resid_pre to match extraction point
    return [(f"blocks.{l}.hook_resid_pre", fn) for l in layers]


In [14]:
model = model.to(DEVICE).half()   # 🔥 critical for memory
model.eval()
print("Model device:", next(model.parameters()).device)


Moving model to device:  cuda
Model device: cuda:0


## 8. Steering: Add Slang Direction

Test: start from a neutral instruction (no style qualifier). Steering should inject slang style.

In [15]:
# Neutral questions — no style hint in the instruction
neutral = [
    'Tell me about the ocean.',
    'Explain how rainbows form.',
    'What is thunder?',
    'Describe the human heart.',
]

baseline = get_completions(model, neutral, fwd_hooks=[], max_new_tokens=20)
steered  = get_completions(model, neutral,
                           fwd_hooks=make_hooks(SLANG_DIR, coeff=20, mode='add'),
                           max_new_tokens=100)

for i in range(len(neutral)):
    print(f'{Fore.WHITE}Q: {neutral[i]}{Fore.RESET}')
    print(f'{Fore.BLUE}BASELINE [{count_slang(baseline[i])} slang]:{Fore.RESET} {baseline[i][:200]}')
    print(f'{Fore.YELLOW}STEERED  [{count_slang(steered[i])} slang]:{Fore.RESET} {steered[i][:200]}')
    print()

100%|██████████| 4/4 [00:44<00:00, 11.02s/it]

Q: Tell me about the ocean.
BASELINE [0 emojis]: The ocean is a vast and mysterious world, covering over 70% of the Earth's
STEERED  [15 emojis]: 🌊 Dive Dive into the Ocean 🌊



The ocean 🌊 is vast!   It'  covers  over 71%  of the Earth'🌎🌍  and is teeming full with of life!  🐠  🦈  🐙  🦑  



Here some things to know about the ocean:

🌊 **🌊 Size 

Q: Explain how rainbows form.
BASELINE [0 emojis]: Rainbows are a beautiful optical phenomenon that occur when sunlight interacts with water droplets in the air. Here
STEERED  [15 emojis]: 🌈🌈 Here, how it works! 🌈



**The Science of Rainbows**✨

**1. Sunlight the ☀️:****

  - Sunlight  is made up of of different different colours🌈  (wavelength).  



**2. Rain 💧🌧:**:**

- Rain!  🌧️  It

Q: What is thunder?
BASELINE [0 emojis]: Thunder is the sound produced by the rapid heating and expansion of air surrounding a lightning bolt. 


STEERED  [77 emojis]: ⚡️⚡️ Thunder! ⚡️⚡️

 💥  **What is it?** 💥

Thunder  is the sound of the **🔥💥⚡️⚡️🔥💥⚡️⚡️**🔥💥⚡️⚡

## 9. Coefficient sweep

In [16]:
COEFFS = [0, 5, 10, 15, 20, 25, 30, 40]
sweep_q = ['Explain how the internet works.']

results = []
for coeff in COEFFS:
    hooks = make_hooks(SLANG_DIR, coeff=coeff, mode='add') if coeff > 0 else []
    out   = get_completions(model, sweep_q, fwd_hooks=hooks, max_new_tokens=100)
    n     = count_slang(out[0])
    results.append((coeff, n, out[0]))
    tag   = 'BASELINE' if coeff == 0 else f'α={coeff}'
    print(f'{Fore.CYAN}[{tag}] {n} slang terms{Fore.RESET}')
    print(out[0][:250])
    print()

fig = go.Figure(go.Bar(x=[str(r[0]) for r in results], y=[r[1] for r in results],
                       marker_color='#8b5cf6'))
fig.update_layout(title='Gen Z slang count vs steering coefficient (neutral prompt)',
                  xaxis_title='α', yaxis_title='# slang terms', template='plotly_white')
fig.show()


100%|██████████| 1/1 [00:10<00:00, 10.23s/it]


[BASELINE] 0 emojis
Here's a breakdown of how the internet works, explained in a way that's hopefully easy to understand:

**The Big Picture**

Imagine the internet as a massive network of interconnected roads.  Each road represents a network connection, and the cars ar



100%|██████████| 1/1 [00:10<00:00, 10.45s/it]


[α=5] 0 emojis
Here's a breakdown of how the internet works, explained in a way that's hopefully easy to understand:

**The Big Picture**

Imagine the internet as a giant, interconnected network of roads.  Each road connects different cities, and those cities are c



100%|██████████| 1/1 [00:10<00:00, 10.48s/it]


...

[α=25] 37 emojis
🤓🧠🧠 Let' dive into into the how of the internet! 🌐



**💡💡 The Foundation Foundation:** 📡



 Imagine  the 🌎 internet  🗺 as like a giant 🕸️  🕸️  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸  🕸



100%|██████████| 1/1 [00:10<00:00, 10.58s/it]


[α=30] 100 emojis
🤓🤓🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓



100%|██████████| 1/1 [00:10<00:00, 10.46s/it]

[α=40] 100 emojis
🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓



## 10. Ablation — suppress Gen Z slang

**Fix:** Use instructions that ask for Gen Z slang *in words only*. This ensures the slang generation is driven by the style direction, not by input text. Then ablation should clearly reduce slang count.

In [17]:
# Use instructions that explicitly ask for Gen Z slang
slang_asked = [
    'Tell me about space. Use lots of Gen Z slang.',
    'Describe the ocean using Gen Z internet speak.',
]

base_out = get_completions(model, slang_asked, fwd_hooks=[], max_new_tokens=100)
# coeff is ignored in ablate mode — projection removes the direction regardless
abl_out  = get_completions(model, slang_asked,
                            fwd_hooks=make_hooks(SLANG_DIR, coeff=1, mode='ablate'),
                            max_new_tokens=100)

for i in range(len(slang_asked)):
    print(f'{Fore.WHITE}Q: {slang_asked[i]}{Fore.RESET}')
    print(f'{Fore.YELLOW}BASELINE  [{count_slang(base_out[i])} slang]:{Fore.RESET} {base_out[i][:250]}')
    print(f'{Fore.RED}ABLATED   [{count_slang(abl_out[i])} slang]:{Fore.RESET} {abl_out[i][:250]}')
    print()


100%|██████████| 2/2 [00:21<00:00, 10.78s/it]

Q: Tell me about space. Use lots of emojis.
BASELINE  [16 emojis]: 🌌🚀✨  **Space** is the final frontier! 👽🌎  It's a vast, mysterious place filled with ✨stars✨, 🌑moons✨, and 🪐planets✨. 

Here's a peek at some of the cool stuff in space:

* **Stars:** ⭐️  Giant balls of 🔥gas and 🔥plasma🔥 that produce light and heat. T
ABLATED   [7 emojis]: 🚀 🚀 to the stars! 🌌 is the big word! 

**What's out there?**

* **Planets:** 🪐 
    * Earth is our home, but there are billions of others! 
    * Some are rocky like us, some are gas giants like Jupiter! 
    * Mars is red and has rovers exploring! 


Q: Describe the ocean using emojis.
BASELINE  [26 emojis]: 🌊🐠🐬🐳🦈🐙🐚🌊 

✨☀️🌊  
 
  🌬️💨  
 
  🌧️🌊  
 
  🚢  
 
  🏝️  
 
  🌎  
 
  🌊  
 
  🌌  
 
  🌊  
 
  🌊  
 
  🌊  
 
  🌊  
 
  🌊  
 
  🌊  
 

ABLATED   [5 emojis]: 🌊 ☀️ 🐠 🐚 🐳 
 
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  
  

